In [1]:
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
from PIL import Image
from torch.optim import Adam,SGD,lr_scheduler
import matplotlib.pyplot as plt
from torchvision import transforms, models
from utils import DiceLoss, SquarePad, SquarePad255
from Water_Dataset import Water_Dataset
from torch.utils.data import Dataset, Subset, DataLoader,random_split

device = 'cuda:1'

group_spec={
    'perfe':114,
    'poly+':152,
    'poly-':408,
    'rough':618,
    'bbox_msk':1113,
    'sam_box':1113,
    'point_msk':int(114*9.76168224/0.96729),
    'scrib_sam':1236,
    'scrib_sam_v3':1236
    
}


mult_spec={
    'perfe':[1,2,4,8,16,32],
    'poly+':[1,2,4,8,16,3648/group_spec['poly+']],
    'poly-':[1,2,4,8,3648/group_spec['poly-']],
    'rough':[1,2,4,3648/group_spec['rough']],
    'bbox_msk':[1,2,3648/group_spec['bbox_msk']],
    'sam_box':[1,2,3648/group_spec['sam_box']],
    'point_msk':[1,2,3648/group_spec['point_msk']],
    'scrib_sam':[1,2,3648/group_spec['scrib_sam']],
    'scrib_sam_v3':[1,2,3648/group_spec['scrib_sam_v3']]
}


base_epoch = 100
pretrained = 'pb'
assert pretrained in ['pb','fb']



img_preprocess = transforms.Compose([
    transforms.RandomEqualize(),
    transforms.ColorJitter(brightness=.5, hue=.3),
#     SquarePad(),
    transforms.CenterCrop(512),
#     transforms.Resize(256),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],\
                         std=[0.229, 0.224, 0.225]),
])
msk_preprocess = transforms.Compose([
#     SquarePad255(),
#     transforms.Resize(256, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.CenterCrop(512),
])


num_classes = 2
loss_fn = nn.CrossEntropyLoss(ignore_index=255) #DiceLoss()

In [2]:
for label_type in ['scrib_sam_v3']:#['perfe','poly+','poly-','rough','bbox_msk','sam_box']:
    DATA = Water_Dataset('./Water/','trainval',label_type, img_preprocess, msk_preprocess)
    group_size = group_spec[label_type]
    for multiplicity in mult_spec[label_type]:

        model = models.segmentation.deeplabv3_resnet50(num_classes=num_classes).to(device)
        #short for DeepLab-VOC-{label_type}-m{multiplicity}
        model_name = f'DLab_H{label_type}_m{round(multiplicity,2)}_{pretrained}' 

        train_subset = Subset(DATA, list(range(round((group_size*multiplicity)))))
        val_subset = Subset(DATA, list(range(3648, len(DATA))))
        train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, drop_last=True,num_workers=16,pin_memory=True)
        val_loader = DataLoader(val_subset, batch_size=16, shuffle=False, drop_last=True,num_workers=16,pin_memory=True)

        epochs=round(base_epoch*(1.5**np.log2(3648/len(train_subset))))

        min_loss = np.inf
        fin_epoch = 0

        if pretrained == 'pb':
            optimizer = Adam(model.parameters(),lr=1e-3,weight_decay=1e-6)
            try:
                checkpoint = torch.load(f'./model_checkpoints/{model_name}.pth')
                fin_epoch = checkpoint['fin_epoch']
                min_loss = checkpoint['min_loss']
                model.load_state_dict(checkpoint['model_state_dict'])
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

            except:
                print(f'new model training {model_name}')

        elif pretrained == 'fb':
            optimizer = SGD(model.module.classifier.parameters(),lr=1e-2,momentum=0.9,weight_decay=5e-4)
            scheduler = lr_scheduler.MultiStepLR(optimizer, milestones=[3000*i for i in range(1,5)], gamma=0.1)
            try:
                checkpoint = torch.load(f'./model_checkpoints/{model_name}.pth')
                fin_epoch = checkpoint['fin_epoch']
                min_loss = checkpoint['min_loss']
                model.load_state_dict(checkpoint['model_state_dict'])
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                print('scheduler loaded')
            except:
                print(f'new model training {model_name}')

        else:
            assert False

        pbar = tqdm(range(epochs-fin_epoch))
        for e in pbar:

            train_loss = 0
            val_loss = 0
            model.train()
            for idx,(X,y) in enumerate(train_loader):

                y = y.squeeze()

                optimizer.zero_grad()
                yhat = model(X.contiguous().to(device))['out']
                loss = loss_fn(yhat,y.to(device))
                loss.backward()
                optimizer.step()
                if pretrained == 'fb':
                    scheduler.step()
                train_loss += loss.item()
            train_loss /= (idx+1)

            model.eval()
            with torch.no_grad():
                class_intersect = np.zeros((num_classes,),dtype='float')
                class_union= np.zeros((num_classes,),dtype='float')

                for idx,(X,y) in enumerate(val_loader):

                    y = y.flatten().to(device)
                    yhat = model(X.contiguous().to(device))['out']
                    yhat_lab = torch.argmax(yhat, dim=1).flatten()
                    yhat_lab[y==255] = 255

                    for j in range(num_classes):

                        y_bi = y == j
                        yhat_bi = yhat_lab == j
                        I = (y_bi * yhat_bi).sum()
                        U = y_bi.sum() + yhat_bi.sum() - I
                        assert I <= U
                        class_intersect[j] += I.item()
                        class_union[j] += U.item()
                IOUs = class_intersect/class_union
                val_loss=-np.mean(IOUs)
    #             print('Train loss',train_loss, 'IOU', -val_loss)

            pbar.set_description(f'DSC loss: train_mean={train_loss}|val={val_loss}')

            if e +fin_epoch+1 <10:
                continue

            if val_loss < min_loss:
                min_loss = val_loss
                to_save ={
                    'min_loss': min_loss,
                    'fin_epoch': e+fin_epoch+1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict()}
                if pretrained=='fb':
                    to_save['scheduler_state_dict']= scheduler.state_dict()

                torch.save(to_save, f'./model_checkpoints/{model_name}.pth')  

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/yz696/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|█████████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:01<00:00, 80.8MB/s]
/tmp/ipykernel_806835/3684176267.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `

new model training DLab_Hscrib_sam_v3_m1_pb


DSC loss: train_mean=0.05735554379205425|val=-0.7644876511634624: 100%|████████████████| 188/188 [1:39:46<00:00, 31.84s/it]


new model training DLab_Hscrib_sam_v3_m2_pb


DSC loss: train_mean=0.05016307317494572|val=-0.7973739870167982: 100%|████████████████| 126/126 [2:05:59<00:00, 60.00s/it]


new model training DLab_Hscrib_sam_v3_m2.95_pb


DSC loss: train_mean=0.038493482101904716|val=-0.8096053153201577: 100%|███████████████| 100/100 [2:24:55<00:00, 86.96s/it]


In [3]:
#Version 1: 512 by 512 patch
pretrained='pb'

performance={}
num_classes=2

test_preprocess = transforms.Compose([
    SquarePad(),
    transforms.CenterCrop(512),
#     transforms.Resize(256),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],\
                         std=[0.229, 0.224, 0.225]),
])
tmsk_preprocess = transforms.Compose([
    SquarePad(),
    transforms.CenterCrop(512),
#     transforms.Resize(256, interpolation=transforms.InterpolationMode.NEAREST),
])

test_loader = DataLoader(Water_Dataset('./Water/','test',label_type, test_preprocess, tmsk_preprocess), 
                         batch_size=32, num_workers=16)


for label_type in ['scrib_sam_v3']: #mult_spec.keys():
    performance[label_type]={}
    
    for multiplicity in mult_spec[label_type]:
        
        #The resnet weight is loaded by default
        model = models.segmentation.deeplabv3_resnet50(num_classes=num_classes)
        #if randomly init, randomly re-initilaize backbone
        model = model.to(device)
        
        model_name = f'DLab_H{label_type}_m{round(multiplicity,2)}_{pretrained}'  
        checkpoint = torch.load(f'./model_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        
        model.eval()
        print(f'Working on label_type={label_type}:{round(multiplicity,2)}')
        class_intersect = np.zeros((num_classes, ),dtype='float')
        class_union = np.zeros((num_classes, ),dtype='float')
    
        with torch.no_grad():
            for idx,(X,y) in enumerate(test_loader):
#                 assert False
                y = y.to(device).contiguous().flatten()
                
                yhat = model(X.contiguous().to(device))['out']
                yhat_lab = torch.argmax(yhat, dim=1).flatten()
                yhat_lab[y == 255] = 255

                for j in range(num_classes):

                    y_bi = y == j
                    yhat_bi = yhat_lab == j
                    I = ((y_bi * yhat_bi).sum()).item()
                    U = (y_bi.sum() + yhat_bi.sum() - I).item()
                    assert I <= U
                    class_intersect[j] += I
                    class_union[j] += U
            
        performance[label_type][multiplicity]=(class_intersect, class_union)
        
np.save(f'Water_Inter-Union_{pretrained}_scrib', performance)


/tmp/ipykernel_806835/1534324699.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f'./model_checkpoints/{model_name}.pth')


Working on label_type=scrib_sam_v3:1
Working on label_type=scrib_sam_v3:2
Working on label_type=scrib_sam_v3:2.95


In [4]:
for key, value in performance['scrib_sam_v3'].items():
    print(key, value[0][1]/value[1][1])

1 0.6449384512080222
2 0.6950827850598069
2.9514563106796117 0.7024258464640624


In [5]:
assert False

AssertionError: 